In [ ]:
import ee

# Authenticate (opens a sign-in flow)
ee.Authenticate()

# Initialize with your project
ee.Initialize(project='even-archway-473806-j5')

In [ ]:
# Define the area of interest as a rectangle around the Feni river stretch
# Format: [min_longitude, min_latitude, max_longitude, max_latitude]
aoi = ee.Geometry.Rectangle([91.30, 23.10, 91.50, 23.30]) #area of interest

# Define the two date windows
before_start = '2024-08-01'
before_end   = '2024-08-15'

after_start  = '2024-08-22'
after_end    = '2024-08-31'

print("Area and dates defined.")

Area and dates defined.


In [ ]:
# Load the Sentinel-1 radar collection
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(aoi) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .select('VV')

# Filter to the two date windows and take the median of each
before = s1.filterDate(before_start, before_end).median().clip(aoi)
after  = s1.filterDate(after_start, after_end).median().clip(aoi)

# Check how many images are in each window
before_count = s1.filterDate(before_start, before_end).size().getInfo()
after_count  = s1.filterDate(after_start, after_end).size().getInfo()

print("Before-flood images found:", before_count)
print("After-flood images found:", after_count)

Before-flood images found: 3
After-flood images found: 2


In [ ]:
import geemap

# Create an interactive map centered on your area
Map = geemap.Map()
Map.centerObject(aoi, 12)

# Radar display settings (VV values are in dB, roughly -25 to 0)
radar_vis = {'min': -25, 'max': 0}

# Add the before and after radar images
Map.addLayer(before, radar_vis, 'Before flood (radar)')
Map.addLayer(after, radar_vis, 'After flood (radar)')

Map

Map(center=[23.200006662814516, 91.40000000000082], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Compute the difference: after minus before (in dB)
difference = after.subtract(before)

# Display the difference
# Blue-ish (negative) = got darker; Red-ish (positive) = got brighter
diff_vis = {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}

Map2 = geemap.Map()
Map2.centerObject(aoi, 12)
Map2.addLayer(difference, diff_vis, 'Change (after - before)')
Map2

Map(center=[23.200006662814516, 91.40000000000082], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Flag pixels that brightened significantly (likely flooded vegetation/cropland)
# Threshold of 3 dB brightening is a common starting point
flood = difference.gt(3)

# Keep only the flooded pixels (mask out the rest)
flood_only = flood.updateMask(flood)

# Display: show the flood in red over the 'before' radar for context
Map3 = geemap.Map()
Map3.centerObject(aoi, 12)
Map3.addLayer(before, {'min': -25, 'max': 0}, 'Before flood (radar)')
Map3.addLayer(flood_only, {'palette': ['red']}, 'Detected flooding')
Map3

Map(center=[23.200006662814513, 91.39999999999883], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Calculate the flooded area in square kilometres
# Each flooded pixel's area, summed over the region
pixel_area = flood.multiply(ee.Image.pixelArea())

flood_area = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
).getInfo()

# Convert from square metres to square kilometres
area_sqkm = flood_area['VV'] / 1e6
print("Detected flooded area:", round(area_sqkm, 2), "square km")

Detected flooded area: 83.43 square km


In [ ]:
# Final map: before radar as context, flooding highlighted in red
final = geemap.Map()
final.centerObject(aoi, 12)
final.addLayer(before, {'min': -25, 'max': 0}, 'Before flood (radar)')
final.addLayer(flood_only, {'palette': ['red']}, 'Detected flooding')
final.addLayerControl()
final

Map(center=[23.200006662814513, 91.39999999999883], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Open a map with drawing tools so you can draw your study area
draw_map = geemap.Map()
draw_map.centerObject(aoi, 11)
draw_map.add_basemap('HYBRID')   # satellite + labels, so you can see the river and border
draw_map

Map(center=[23.200006662814516, 91.40000000000082], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Grab the polygon you drew
aoi = draw_map.user_roi

# Confirm it worked
print("New study area captured:", aoi.getInfo()['type'])

New study area captured: Polygon


In [ ]:
# Re-run the whole flood detection on the new polygon area
before = s1.filterDate(before_start, before_end).median().clip(aoi)
after  = s1.filterDate(after_start, after_end).median().clip(aoi)

difference = after.subtract(before)
flood = difference.gt(3)
flood_only = flood.updateMask(flood)

# Recalculate flooded area for the new region
pixel_area = flood.multiply(ee.Image.pixelArea())
flood_area = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
).getInfo()

area_sqkm = flood_area['VV'] / 1e6
print("Updated flooded area:", round(area_sqkm, 2), "square km")

Updated flooded area: 193.02 square km


In [ ]:
# Check the size of your drawn study area (with the required error margin)
area_of_aoi = aoi.area(maxError=1).getInfo() / 1e6
print("Your study area is:", round(area_of_aoi, 2), "square km")

Your study area is: 1149.96 square km


In [ ]:
# Fresh map to redraw a tighter study area
draw_map2 = geemap.Map()
draw_map2.setCenter(91.40, 23.20, 12)   # centered on your river stretch, zoomed in
draw_map2.add_basemap('HYBRID')
draw_map2

Map(center=[23.2, 91.4], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
# Capture the new tighter polygon
aoi = draw_map2.user_roi

# Check its size right away
area_of_aoi = aoi.area(maxError=1).getInfo() / 1e6
print("New study area is:", round(area_of_aoi, 2), "square km")

New study area is: 441.8 square km


In [ ]:
# Re-run flood detection on the new tighter polygon
before = s1.filterDate(before_start, before_end).median().clip(aoi)
after  = s1.filterDate(after_start, after_end).median().clip(aoi)

difference = after.subtract(before)
flood = difference.gt(3)
flood_only = flood.updateMask(flood)

# Recalculate flooded area
pixel_area = flood.multiply(ee.Image.pixelArea())
flood_area = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
).getInfo()

area_sqkm = flood_area['VV'] / 1e6
print("Flooded area in study region:", round(area_sqkm, 2), "square km")
print("Study area total:", round(aoi.area(maxError=1).getInfo() / 1e6, 2), "square km")
print("Percent flooded:", round(area_sqkm / (aoi.area(maxError=1).getInfo() / 1e6) * 100, 1), "%")

Flooded area in study region: 75.21 square km
Study area total: 441.8 square km
Percent flooded: 17.0 %


In [ ]:
# Final flood map
final = geemap.Map()
final.centerObject(aoi, 12)
final.addLayer(before, {'min': -25, 'max': 0}, 'Before flood (radar)')
final.addLayer(flood_only, {'palette': ['red']}, 'Detected flooding')
final.add_layer_control()
final

Map(center=[23.09686801281522, 91.4571683421227], controls=(WidgetControl(options=['position', 'transparent_bg…